In [ ]:
library(shazam)
library(alakazam)
library(readr)
library(dplyr)
library(ggplot2)

In [ ]:
# BASELINE MODEL
# 1. Cargar archivo
archivo_clones <- "../data/output/repertorio_A_insilico_100_seqs_clone-pass.tsv"
db <- read_tsv(archivo_clones) %>%
  mutate(
    sample_id = "repertorio_naive",
    clone_id = as.character(clone_id)
  ) %>%
  filter(!is.na(clone_id))

# 2. Asignar IGHM si c_call está vacío
if (all(is.na(db$c_call))) {
  db$c_call <- "IGHM"
}

# 3. Colapsar clones
clones <- collapseClones(
  db,
  cloneColumn="clone_id",
  sequenceColumn="sequence_alignment",
  germlineColumn="germline_alignment",
  regionDefinition=IMGT_V,
  method="thresholdedFreq",
  minimumFrequency=0.3,
  includeAmbiguous=FALSE,
  breakTiesStochastic=FALSE,
  nproc=1
)

# 4. Calcular selección
baseline <- calcBaseline(
  clones,
  testStatistic="focused",
  regionDefinition=IMGT_V,
  nproc=1
)

# 5. Mensaje pipeline con pdfs o no
if (length(baseline) == 0) {
  message("Pipeline ejecutado correctamente. No hay PDFs de selección porque el repertorio naïve no tiene mutaciones.")
} else {
  message("Pipeline ejecutado correctamente. Hay resultados de selección disponibles.")
}

grouped <- groupBaseline(baseline, groupBy="sample_id")

# Chequear que grouped tenga datos y la columna sample_id
if (!is.null(grouped) && nrow(grouped) > 0 && "sample_id" %in% colnames(grouped)) {
  plotBaselineSummary(grouped, idColumn="sample_id")
  plotBaselineDensity(grouped, idColumn="sample_id")
} else {
  message("No hay PDFs de selección → no se genera gráfico.")
}

In [ ]:
write_tsv(grouped, "../results/shm_models/baseline/repertorio_A_100seq_baseline.tsv")